In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import LearningRateScheduler

In [ ]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 2  # Normal and Pneumonia

In [ ]:
# Data Generators with Data Augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

In [ ]:
train_generator = train_datagen.flow_from_directory(
    '/content/drive/MyDrive/Major_Project_Pneumonia_Detection/Dataset_2/train',
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

Found 5276 images belonging to 2 classes.


In [ ]:
test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = test_datagen.flow_from_directory(
    '/content/drive/MyDrive/Major_Project_Pneumonia_Detection/Dataset_2/test',
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

Found 624 images belonging to 2 classes.


In [ ]:
validation_datagen = ImageDataGenerator(rescale=1./255)
validation_generator = validation_datagen.flow_from_directory(
    '/content/drive/MyDrive/Major_Project_Pneumonia_Detection/Dataset_2/val',
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

Found 16 images belonging to 2 classes.


In [ ]:
from tensorflow.keras.layers import Dropout

# Load Pretrained DenseNet-169 Model
base_model = DenseNet121(weights='imagenet', include_top=False, input_shape=(224, 224, 3))


# Freeze the base model layers
base_model.trainable = False

# Add Global Average Pooling Layer
x = base_model.output
x = GlobalAveragePooling2D()(x)

predictions = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

29084464/29084464 [==============================] - 2s 0us/step


In [ ]:
# # Compile the Model
# model.compile(optimizer=Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])

# # Calculate steps_per_epoch and validation_steps
# steps_per_epoch = len(train_generator)
# validation_steps = len(validation_generator)

# # Learning Rate Scheduler
# def lr_schedule(epoch):
#     initial_learning_rate = 0.0001
#     decay = 0.9
#     if epoch % 2 == 0:
#         return initial_learning_rate * decay
#     else:
#         return initial_learning_rate

# lr_callback = LearningRateScheduler(lr_schedule)

# # Train the Model with Correct steps_per_epoch and validation_steps
# model.fit(
#     train_generator,
#     steps_per_epoch=steps_per_epoch,
#     epochs=10,
#     validation_data=validation_generator,
#     validation_steps=validation_steps,
#     callbacks=[lr_callback]
# )

In [ ]:
# Save the Model
model.load_weights('/content/drive/MyDrive/Major_Project_Pneumonia_Detection/models/pneumonia_model.h5')